# Framecount analysis — 20260820 M66741 - exp w1 D2

Reads `cameraTop_framecount` and `cameraFront_framecount` CSVs from the data folder.

In [ ]:
import glob
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

data_dir = Path("../data/20260820 M66741 - exp w1 D2")

In [ ]:
top_path = glob.glob(str(data_dir / "cameraTop_framecount*.csv"))[0]
front_path = glob.glob(str(data_dir / "cameraFront_framecount*.csv"))[0]

top_framecount = pd.read_csv(top_path, header=None, names=["frame_count"])
front_framecount = pd.read_csv(front_path, header=None, names=["frame_count"])

print(f"cameraTop_framecount: {len(top_framecount)} rows")
print(f"cameraFront_framecount: {len(front_framecount)} rows")

In [ ]:
top_framecount.head()

In [ ]:
front_framecount.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=False)

axes[0].plot(top_framecount["frame_count"])
axes[0].set_title("cameraTop_framecount")
axes[0].set_xlabel("row index")
axes[0].set_ylabel("frame count")

axes[1].plot(front_framecount["frame_count"])
axes[1].set_title("cameraFront_framecount")
axes[1].set_xlabel("row index")
axes[1].set_ylabel("frame count")

fig.tight_layout()
plt.show()

## Cross-camera check using the Harp-timestamped trigger events

The embedded `frame_count` columns above come from each camera's own internal hardware counter (`Metadata.embeddedFrameCounter`), so their absolute values and start points are independent per camera — not directly comparable.

However, `clock_therm.bonsai` shows that `cameraTimestamp*.csv` and `cameraFront*.csv` (the bare, non-`_framecount` files) are the Harp behavior board's `TimestampedCamera0Frame` (Top) and `TimestampedCamera1Frame` (Front) registers. Both are timestamped by the **same Harp clock**, so — despite the generic file naming — these two files *are* directly comparable across cameras, unlike the per-camera embedded counters.

In [ ]:
harp_top_path = glob.glob(str(data_dir / "cameraTimestamp*.csv"))[0]
harp_front_path = glob.glob(str(data_dir / "cameraFront[0-9]*.csv"))[0]

harp_top = pd.read_csv(harp_top_path, header=None, names=["timestamp", "event"])
harp_front = pd.read_csv(harp_front_path, header=None, names=["timestamp", "event"])

print(f"Harp Top (Camera0) events: {len(harp_top)}")
print(f"Harp Front (Camera1) events: {len(harp_front)}")

In [ ]:
top_first, top_last = harp_top["timestamp"].iloc[0], harp_top["timestamp"].iloc[-1]
front_first, front_last = harp_front["timestamp"].iloc[0], harp_front["timestamp"].iloc[-1]

print(f"Top:   {len(harp_top)} events over {top_last - top_first:.3f} s "
      f"({(len(harp_top) - 1) / (top_last - top_first):.3f} Hz)")
print(f"Front: {len(harp_front)} events over {front_last - front_first:.3f} s "
      f"({(len(harp_front) - 1) / (front_last - front_first):.3f} Hz)")
print()
print(f"Start offset (front_first - top_first): {front_first - top_first:.3f} s")
print(f"End offset   (front_last - top_last):   {front_last - top_last:.3f} s")